# Origins-of-life vent campaign

_Investigation `origins-of-life-vent` — coder reproduction notebook._

**Question.** Could the first cells have originated at alkaline hydrothermal vents? Can a physically plausible early-Earth vent–ocean interface generate and sustain, in order, the free energy, reduced carbon, self-amplifying chemistry, concentration, and compartments a protocell needs — without inserting metabolism, membranes, or heredity by assumption?

A staged, falsifiable Vivarium campaign testing the alkaline-vent theory for the origin of cells — treating a cellular origin as coupled transitions from geochemistry to bioenergetic autonomy, grounded in serpentinization-driven alkaline hydrothermal vents, with explicit competing scenarios.

---

**How to use this notebook**

1. **Run ▸ Run All Cells** to reproduce every study end to end.
2. To explore, edit a study's **Parameters** cell, then re-run from its **Run** section — the figures regenerate with your values.
3. The **Setup** cell at the top is collapsed scaffolding (environment + render helpers); expand it only to inspect. Set `RERUN = False` there to render the committed `runs.db` without re-simulating.

The prose states the **question and parameters**; the figures each run produces are the results.


**Competing frameworks**

- **Classical alkaline-vent (Russell/Martin/Lane/Sojo)**
- **2026 unfavorable reassessment (Tutolo; Jackson)**
- **Shallow-vent (Barge & Price)**
- **Terrestrial geothermal (Mulkidjanian)**
- **Cyanosulfidic/photochemical (Sutherland)**


In [ ]:
"""Setup — environment & helpers (runs automatically; open only to inspect).

Generated by vivarium-workbench (notebook_export). This cell resolves the repo,
registers the workspace Core, and defines the render helpers; the science lives
in the sections below. Each study is re-run live with the workspace's own
process-bigraph protocol and its figures rendered from the resulting runs.db.
"""
import os
import sys
from pathlib import Path

# Resolve the repository root robustly so this notebook runs from a fresh clone
# at ANY path with no setup (no env var, no path editing). Priority:
#   1. $VIVARIUM_REPO, if it points at a real directory;
#   2. walk up from the notebook's working directory for the repo markers
#      (a directory holding both 'workspace/' and 'pyproject.toml') — Jupyter
#      starts in the notebook's dir, so a committed notebook finds its own root;
#   3. the absolute path it was generated for (back-compat for old layouts);
#   4. the current working directory (last resort).
def _find_repo_root(_start):
    for _cand in (_start, *_start.parents):
        if (_cand / "workspace").is_dir() and (_cand / "pyproject.toml").is_file():
            return _cand
    return None

REPO = None
_env = os.environ.get("VIVARIUM_REPO")
if _env and Path(_env).is_dir():
    REPO = Path(_env)
if REPO is None:
    REPO = _find_repo_root(Path.cwd().resolve())
if REPO is None and Path('/home/runner/work/viva-genesis/viva-genesis').is_dir():
    REPO = Path('/home/runner/work/viva-genesis/viva-genesis')
if REPO is None:
    REPO = Path.cwd()
sys.path.insert(0, str(REPO))
# Composite specs use repo-root-relative paths (datasets, caches), and the
# workspace's runner/renderer assume cwd == repo root — so run from there.
os.chdir(REPO)

# Re-simulate from scratch? Set False to render the committed runs.db (fast).
RERUN = True

# Keep the output clean: composite discovery and the pydantic models emit
# benign UserWarnings (optional packages it skips, a field-name shadow) and the
# simulators emit DeprecationWarnings — none are actionable from a reproduction
# notebook, so silence those two categories. Remove this to see them.
import warnings as _warnings
_warnings.filterwarnings("ignore", category=UserWarning)
_warnings.filterwarnings("ignore", category=DeprecationWarning)

# --- standard process-bigraph protocol: register the workspace's Core ---
from viva_genesis.core import build_core
core = build_core()

# --- imported from the repo this notebook was generated for ---

from IPython.display import HTML, display

import contextlib as _contextlib, io as _io
@_contextlib.contextmanager
def quiet():
    """Silence the simulator's verbose per-step stdout so the notebook
    output stays readable (the figures below are the results)."""
    with _contextlib.redirect_stdout(_io.StringIO()):
        yield

import html as _htmlmod
def show_viz(_h, height=560):
    """Display a visualization's HTML in an isolated iframe.

    The figures embed their own scripts (e.g. Plotly); JupyterLab does not
    execute <script> tags from display(HTML(...)), so an iframe srcdoc is
    used instead — the browser runs the scripts inside the frame."""
    display(HTML(
        '<iframe srcdoc="{}" style="width:100%;height:{}px;border:0">'
        '</iframe>'.format(_htmlmod.escape(_h, quote=True), height)
    ))

import json as _json
def describe_spec(spec):
    """Print a composite spec's structure (parameters, processes, wiring)
    then the full editable dict. The spec is plain data — assign to any
    field (e.g. spec['state'][proc]['config'][...]) before building."""
    print("composite:", spec.get("name"))
    if spec.get("description"):
        print("description:", str(spec["description"]).strip())
    _params = spec.get("parameters") or {}
    if _params:
        print("\nparameters (filled into ${name} placeholders):")
        for _p, _pdef in _params.items():
            print(f"  {_p}: default={_pdef.get('default')!r}  type={_pdef.get('type')}")
    print("\nprocesses (node -> address):")
    for _node, _body in (spec.get("state") or {}).items():
        if not (isinstance(_body, dict) and _body.get("_type") == "process"):
            continue
        print(f"  {_node}  ->  {_body.get('address')}   interval={_body.get('interval')!r}")
        for _port in ("inputs", "outputs"):
            if _body.get(_port):
                print(f"      {_port} ports: {_body[_port]}")
    print("\nfull editable spec dict:")
    print(_json.dumps(spec, indent=2, default=str))

def materialize_composite(spec_id, overrides=None):
    """Build a composite into an editable payload {name, description,
    parameters, state, _overrides, _id}. Works for @composite_generator
    functions via the process-bigraph registry (the common case); ``state``
    is the built document (process nodes at top level)."""
    import importlib
    from process_bigraph.composite_generator import (
        _REGISTRY, apply_core_extensions, build_generator)
    try:
        from process_bigraph.composite_generator import discover_generators
        discover_generators()
    except Exception:
        pass
    entry = _REGISTRY.get(spec_id)
    if entry is None:
        # editable installs are invisible to packages_distributions, so
        # discover alone can miss them — prime the module the id names.
        try:
            importlib.import_module(spec_id.rsplit('.', 1)[0])
            from process_bigraph.composite_generator import discover_generators as _dg
            _dg()
        except Exception:
            pass
        entry = _REGISTRY.get(spec_id)
    if entry is None:
        raise KeyError('composite not registered: ' + repr(spec_id))
    params = dict(entry.parameters or {})
    ov = {k: (v.get('default') if isinstance(v, dict) else None)
          for k, v in params.items()}
    if overrides:
        ov.update({k: val for k, val in overrides.items() if k in params})
    _c = apply_core_extensions(entry, core)
    try:
        doc = build_generator(entry, ov or None, core=_c)
    except TypeError:
        doc = entry.func(_c, **(ov or {}))
    return {'name': entry.name, 'description': entry.description,
            'parameters': params, 'state': doc, '_overrides': ov, '_id': spec_id}

import base64 as _b64, pathlib as _pl
_STATIC_VIZ_SCHEMES = ('image:', 'file:', 'gif:', 'png:', 'svg:', 'jpg:', 'jpeg:')
def _render_viz_process(address, config):
    """Render a Visualization process/step (e.g. ``local:<Class>``) to
    HTML: wrap it as a one-step composite and run it once — the viz writes
    Plotly HTML to its ``html`` output port. Mirrors the workbench's own
    env-worker viz path. Self-running vizes need no run data."""
    from process_bigraph import Composite
    _doc = {
        'inputs_store': {}, 'output_store': '',
        'visualization': {
            '_type': 'step', 'address': address,
            'config': {k: v for k, v in (config or {}).items()
                       if k not in ('inputs_map', 'sources')},
            'inputs': {}, 'outputs': {'html': ['output_store']},
        },
    }
    _comp = Composite({'state': _doc}, core=core)
    _comp.run(1)
    _html = _comp.state.get('output_store')
    if isinstance(_html, dict):
        _html = _html.get('value') or _html.get('_value') or ''
    return _html if isinstance(_html, str) else ''
def _render_one(address, config, runs_db, study_yaml, overrides=None):
    """Generic figure renderer (no workspace render_study_viz.py).
    A static ``image:``/``file:`` (or bare-path) address embeds the file;
    any other address (``local:<VizProcess>``) is run as a Visualization
    and its HTML returned. ``overrides`` (the composite's edited parameter
    dict) is merged into the viz config first so a self-running figure
    reflects edited parameters; a viz that does not accept those keys
    falls back to the base config."""
    addr = str(address or '')
    _head = addr.split('/', 1)[0]
    _is_static = (not addr) or (':' not in _head) or addr.startswith(_STATIC_VIZ_SCHEMES)
    if not _is_static:
        if core is None:
            return f'<p style="color:#b91c1c">no core to render {address}</p>'
        _base = {k: v for k, v in (config or {}).items()}
        _tries = ([{**_base, **overrides}] if overrides else []) + [_base]
        _err = ''
        for _cfg in _tries:
            try:
                _h = _render_viz_process(addr, _cfg)
            except Exception as _e:
                _err = (f'<p style="color:#b91c1c">visualization {address} not renderable yet: '
                        f'{type(_e).__name__}: {_e}</p>')
                continue
            return _h or f'<p style="color:#6b7280">{address} produced no HTML</p>'
        return _err
    for _scheme in _STATIC_VIZ_SCHEMES:
        if addr.startswith(_scheme):
            addr = addr[len(_scheme):]; break
    _p = _pl.Path(addr)
    if not _p.is_absolute():
        _p = _pl.Path(study_yaml).resolve().parent / _p
    if not _p.is_file():
        return f'<p style="color:#b91c1c">figure not found: {address}</p>'
    _suffix = _p.suffix.lower()
    if _suffix == '.svg':
        return _p.read_text(encoding='utf-8', errors='replace')
    if _suffix in ('.png', '.jpg', '.jpeg', '.gif', '.webp'):
        _mime = 'jpeg' if _suffix in ('.jpg', '.jpeg') else _suffix[1:]
        _data = _b64.b64encode(_p.read_bytes()).decode('ascii')
        return f'<img src="data:image/{_mime};base64,{_data}" style="max-width:100%"/>'
    if _suffix in ('.html', '.htm'):
        return _p.read_text(encoding='utf-8', errors='replace')
    return f'<p style="color:#6b7280">unsupported figure type: {address}</p>'

## Study: Vent reactor baseline (Campaign 0) (`og-00-vent-reactor-baseline`)

**Question.** Can a Lost City–like, serpentinization-fed vent–ocean interface reproduce observed fluid mixing, heat transport, mineral precipitation and H2/alkalinity delivery under field-constrained conditions — with no origin-of-life claim?

**Objective.** Build a reduced vent–ocean reactor and validate it against Lost City field ranges to establish a geochemical/spatial testbed.

**Hypothesis.** A 1D/2D porous-wall reactor with prescribed boundary fluxes reproduces published Lost City ranges (T, pH, H2, DIC) and qualitative precipitation trends while conserving mass and charge.

**Claim.** A real PHREEQC + FiPy serpentinization reactor settles into the Lost City window (T 51 °C, pH 9.37) with sustained H2 and exact mass/charge/spatial-flux conservation.


### Design — originally proposed

_The mechanisms and methods proposed when this study was designed; the model **actually built** (and the tools it uses) is shown below. These may differ as the design was realized._

**Proposed mechanisms**

- A: serpentinization as boundary flux
- B: flow/mixing/precipitation, evolving porosity

**Proposed models / methods**

- PHREEQC or Reaktoro (speciation + mineral kinetics)
- FiPy finite-volume advection–diffusion–reaction
- prescribed vent source term

**Key assumptions**

- Hadean ocean chemistry parameterized separately — never reuse modern seawater as the ancient prior.
- Partial equilibrium: a prescribed olivine dissolution rate (flow/T-driven) feeds PHREEQC equilibrium speciation each step — fast aqueous/mineral equilibrium vs a rate-limited rock reaction, standard for reactive-transport.
- The heat-partition fraction (fraction of serpentinization enthalpy retained in the fluid parcel) is CALIBRATED to the observed 40-91 C vent temperature; the chemistry (H2, pH, pe, charge) is emergent from PHREEQC, not tuned.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `Vent reactor baseline (Campaign 0)` | `viva_genesis.composites.vent_reactor.serpentinizing_vent_reactor` | 0 | — |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `viva_genesis.composites.vent_reactor.serpentinizing_vent_reactor`**

A reusable alkaline-vent reactor module. Module A = genuine PHREEQC serpentinization (phreeqpython + vendored llnl.dat): olivine dissolution + serpentine/brucite/magnetite equilibrium gives dissolved H2 (reducing power), alkalinity, heat, pe, and a charge-balanced fluid. Module B = genuine FiPy finite-volume advection-diffusion-reaction column transporting those sources across the vent-ocean interface and setting outlet T/pH/porosity. PHREEQC conserves mass+charge and FiPy's FVM conserves flux, satisfying the og-00 contract invariants. Requires phreeqpython + fipy.


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_viva_genesis_composites_vent_reactor_serpentinizing_vent_reactor = {
    'flow_velocity': 0.001,   # Boundary water flux through the reaction zone (m/s)
    'olivine_rate' : 0.15,   # Olivine dissolution rate (mol per flow-saturated reaction step)
    'nx'           : 40,   # FiPy finite-volume cells across the interface column
}

### Typed contract (declared interface)

_The bigraph-schema typed ports + conservation predicates this study's composite must satisfy (audited in the workbench §Contract panel). Declared design — independent of whether a run exists yet._

**Inputs**

| port | type | units | bounds |
| --- | --- | --- | --- |
| `temperature` | float | kelvin | [273.0, 673.0] |
| `pressure` | float | pascal | [100000.0, 50000000.0] |
| `flow_velocity` | float | meter/second | >= 0.0 |
| `H2` | float | molar | >= 0.0 |
| `DIC` | float | molar | >= 0.0 |

**Outputs**

| port | type | units | bounds |
| --- | --- | --- | --- |
| `H2_flux` | float | mole/second | >= 0.0 |
| `alkalinity_flux` | float | mole/second |  |
| `heat_flux` | float | watt |  |
| `porosity` | float | dimensionless | [0.0, 1.0] |

**Conditions**

- `mass_balance` (invariant): `abs(residual_C)+abs(residual_H)+abs(residual_O) < tol`
- `charge_neutrality` (invariant): `abs(net_charge) < tol`
- `spatial_flux_conservation` (invariant): `divergence(flux) == source_minus_sink`
- `energy_directionality` (invariant): `free_energy_flux_logged == true`


### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: og-00-vent-reactor-baseline ===
STUDY = 'og-00-vent-reactor-baseline'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_Vent_reactor_baseline_Campaign_0 = 100
INTERVAL_Vent_reactor_baseline_Campaign_0 = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_genesis_composites_vent_reactor_serpentinizing_vent_reactor if 'ov_viva_genesis_composites_vent_reactor_serpentinizing_vent_reactor' in dir() else None  # edits from the panel cell, if any
            spec_viva_genesis_composites_vent_reactor_serpentinizing_vent_reactor = materialize_composite('viva_genesis.composites.vent_reactor.serpentinizing_vent_reactor', _ov)
            _comp = Composite({'state': spec_viva_genesis_composites_vent_reactor_serpentinizing_vent_reactor['state']}, core=core)
            _comp.run(STEPS_Vent_reactor_baseline_Campaign_0)
        print('ran', 'Vent reactor baseline (Campaign 0)', f'(viva_genesis.composites.vent_reactor.serpentinizing_vent_reactor) for', STEPS_Vent_reactor_baseline_Campaign_0, 'steps')
    except Exception as _e:
        print(f"Vent reactor baseline (Campaign 0) not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**vent-cross-section**


In [ ]:
# vent-cross-section
show_viz(_render_one('local:VentCrossSection', {'composite': 'viva_genesis.composites.vent_reactor.serpentinizing_vent_reactor', 'n_steps': 60, 'title': 'Campaign 0 — real 2-D vent cross-section (FiPy)'}, RUNS_DB, STUDY_YAML, overrides=(ov_viva_genesis_composites_vent_reactor_serpentinizing_vent_reactor if 'ov_viva_genesis_composites_vent_reactor_serpentinizing_vent_reactor' in dir() else None)))

**vent-reactor-timeseries**


In [ ]:
# vent-reactor-timeseries
show_viz(_render_one('local:VentReactorTimeseries', {'composite': 'viva_genesis.composites.vent_reactor.serpentinizing_vent_reactor', 'n_steps': 120, 'title': 'Campaign 0 — real vent reactor (PHREEQC + FiPy)'}, RUNS_DB, STUDY_YAML, overrides=(ov_viva_genesis_composites_vent_reactor_serpentinizing_vent_reactor if 'ov_viva_genesis_composites_vent_reactor_serpentinizing_vent_reactor' in dir() else None)))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| vent-temperature-in-lostcity-range _(primary)_ | kind=derived path=settled_temperature_c detail=last-quartile mean outlet temperature, degrees C | op range low 40.0 high 91.0 detail settled T in [40, 91] C |
| h2-flux-sustained _(primary)_ | kind=derived path=min_sustained_H2_flux detail=minimum H2 source flux over the back half of the run | op > value 1e-09 detail min sustained H2 flux > 0 |
| charge-conserved _(primary)_ | kind=derived path=max_abs_charge_imbalance detail=worst-case |charge imbalance| over the run (eq/kgw) | op <= value 1e-06 detail |charge imbalance| < 1e-6 throughout |
| spatial-flux-conserved _(primary)_ | kind=derived path=max_flux_divergence_residual detail=worst-case normalized FVM flux-divergence residual over the run | op <= value 0.001 detail max |flux divergence residual| < 1e-3 (direct LU solve ~ machine eps) |
| vent-pH-in-lostcity-range _(primary)_ | kind=derived path=settled_pH detail=last-quartile mean outlet pH | op range low 9.0 high 10.8 detail settled pH in [9.0, 10.8] |


### Literature anchors

| expectation | model observable | source | status |
| --- | --- | --- | --- |
| Lost City ~40-91 C, pH ~9-10.8, >=30 ky activity | outlet T,pH; run duration | Kelley 2001 / Fruh-Green 2003 / Ludwig 2006 | Met by the real coupled model: settled T ~51 C and pH ~9.4 are both in the Lost City window, H2 reducing power is sustained, and mass/charge/flux conservation hold. pH was closed by reacting peridotite (olivine + pyroxene) so the Ca-OH endmember buffers the fluid to field pH — a real PHREEQC result. |


## Study: Geology to reduced carbon (Campaign I) (`og-01-geology-to-reduced-carbon`)

**Question.** Which combinations of T, pH, pressure, H2, CO2 and mineralogy generate sustained net organic-carbon flux across the interface — reproducing Hudson/Preiner/Altair regimes before any extrapolation?

**Objective.** Add H2-driven CO2 reduction, reproduce the three benchmark experiments, and map the parameter region of sustained net organic-carbon flux.

**Hypothesis.** Over Fe/Ni/S surfaces and across the pH gradient, H2-driven CO2 reduction yields formate/acetate/pyruvate at lab-calibrated rates within a bounded (T,pH,H2,pressure) region; outside it net flux goes to zero.

**Claim.** H2-driven CO2 reduction over Fe/Ni/S reproduces the Preiner/Hudson lab yields and net organic-carbon flux in a bounded favorable regime; the Tutolo-2026 arm shows the usable proton-motive gradient collapses at in-situ T/P, zeroing the flux.


### Design — originally proposed

_The mechanisms and methods proposed when this study was designed; the model **actually built** (and the tools it uses) is shown below. These may differ as the design was realized._

**Proposed mechanisms**

- C: pH-gradient electrochemistry
- D: mineral-catalyzed carbon fixation

**Proposed models / methods**

- Nernst–Planck–Poisson (electroneutral) on FiPy/FEniCSx
- Cantera/custom heterogeneous kinetics
- eQuilibrator feasibility
- PyMC Bayesian calibration to Preiner/Hudson/Altair

**Key assumptions**

- Encode pH(T,P,ionic_strength), never laboratory pH as a constant environmental state.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `Geology to reduced carbon (Campaign I) — favorable` | `viva_genesis.composites.carbon_fixation.hydrothermal_carbon_fixation` | 0 | — |
| `Tutolo-2026 in-situ (unfavorable arm)` | `viva_genesis.composites.carbon_fixation.hydrothermal_carbon_fixation` | 0 | gradient_efficiency=0.1 |

Declared parameter sets (`study.yaml` variants):

- **tutolo-2026** — `gradient_efficiency=0.1`
- **control-no-gradient** — `gradient_efficiency=1e-06`
- **control-no-h2** — `h2=1e-12`
- **control-no-catalyst** — `catalyst_active=1e-06`


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `viva_genesis.composites.carbon_fixation.hydrothermal_carbon_fixation`**

A reusable H2/pH-driven carbon-fixation module. Module C = genuine FiPy Nernst-Planck pH-gradient electrochemistry (pmf, current density). Module D = genuine eQuilibrator ΔG' feasibility gate + lab-calibrated (Preiner/Hudson) mineral-catalyzed CO2 reduction to formate/acetate/pyruvate. gradient_efficiency=1.0 is the favorable arm; ~0.1 is the Tutolo-2026 in-situ collapse. catalyst_active/h2=0 give the controls. Requires equilibrator_api + fipy.


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_viva_genesis_composites_carbon_fixation_hydrothermal_carbon_fixation = {
    'gradient_efficiency': 1.0,   # Usable fraction of the geochemical ΔpH at in-situ T/P (1.0 favorable; ~0.1 Tutolo-2026 collapse; 0 = no-gradient control)
    'catalyst_active'    : 1.0,   # Active Fe/Ni/S mineral catalyst (0 = no-catalyst control)
    'h2'                 : 0.005,   # Dissolved H2 feed (M); ~0 = no-H2 control
    'co2'                : 0.002,   # Dissolved CO2 feed (M)
    'ph_vent'            : 9.5,   # Alkaline vent-side pH (from og-00)
    'ph_ocean'           : 6.0,   # Acidic early-ocean-side pH
    'dilution_rate'      : 0.0,   # Open-flow washout (1/s); 0 = closed batch (yield studies), >0 = open vent reactor → sustained-throughput regime (phase map)
}

### Typed contract (declared interface)

_The bigraph-schema typed ports + conservation predicates this study's composite must satisfy (audited in the workbench §Contract panel). Declared design — independent of whether a run exists yet._

**Inputs**

| port | type | units | bounds |
| --- | --- | --- | --- |
| `temperature` | float | kelvin | [273.0, 673.0] |
| `pH` | float | dimensionless | [0.0, 14.0] |
| `pressure` | float | pascal | [100000.0, 50000000.0] |
| `H2` | float | molar | >= 0.0 |
| `CO2` | float | molar | >= 0.0 |
| `ionic_strength` | float | molar | >= 0.0 |

**Outputs**

| port | type | units | bounds |
| --- | --- | --- | --- |
| `formate` | float | molar | >= 0.0 |
| `acetate` | float | molar | >= 0.0 |
| `pyruvate` | float | molar | >= 0.0 |
| `current_density` | float | ampere/meter**2 |  |
| `electric_potential` | float | volt |  |
| `net_carbon_flux` | float | mole/second |  |

**Conditions**

- `charge_neutrality` (invariant): `abs(net_charge) < tol`
- `energy_directionality` (invariant): `free_energy_flux_logged == true`
- `mass_balance` (invariant): `abs(residual_C)+abs(residual_H)+abs(residual_O) < tol`


### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: og-01-geology-to-reduced-carbon ===
STUDY = 'og-01-geology-to-reduced-carbon'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_Geology_to_reduced_carbon_Campaign_I_favorable = 100
INTERVAL_Geology_to_reduced_carbon_Campaign_I_favorable = 1.0
STEPS_Tutolo_2026_in_situ_unfavorable_arm = 100
INTERVAL_Tutolo_2026_in_situ_unfavorable_arm = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_genesis_composites_carbon_fixation_hydrothermal_carbon_fixation if 'ov_viva_genesis_composites_carbon_fixation_hydrothermal_carbon_fixation' in dir() else None  # edits from the panel cell, if any
            spec_viva_genesis_composites_carbon_fixation_hydrothermal_carbon_fixation = materialize_composite('viva_genesis.composites.carbon_fixation.hydrothermal_carbon_fixation', _ov)
            _comp = Composite({'state': spec_viva_genesis_composites_carbon_fixation_hydrothermal_carbon_fixation['state']}, core=core)
            _comp.run(STEPS_Geology_to_reduced_carbon_Campaign_I_favorable)
        print('ran', 'Geology to reduced carbon (Campaign I) — favorable', f'(viva_genesis.composites.carbon_fixation.hydrothermal_carbon_fixation) for', STEPS_Geology_to_reduced_carbon_Campaign_I_favorable, 'steps')
    except Exception as _e:
        print(f"Geology to reduced carbon (Campaign I) — favorable not runnable yet: {type(_e).__name__}: {_e}")
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_genesis_composites_carbon_fixation_hydrothermal_carbon_fixation if 'ov_viva_genesis_composites_carbon_fixation_hydrothermal_carbon_fixation' in dir() else None  # edits from the panel cell, if any
            spec_viva_genesis_composites_carbon_fixation_hydrothermal_carbon_fixation = materialize_composite('viva_genesis.composites.carbon_fixation.hydrothermal_carbon_fixation', _ov)
            _comp = Composite({'state': spec_viva_genesis_composites_carbon_fixation_hydrothermal_carbon_fixation['state']}, core=core)
            _comp.run(STEPS_Tutolo_2026_in_situ_unfavorable_arm)
        print('ran', 'Tutolo-2026 in-situ (unfavorable arm)', f'(viva_genesis.composites.carbon_fixation.hydrothermal_carbon_fixation) for', STEPS_Tutolo_2026_in_situ_unfavorable_arm, 'steps')
    except Exception as _e:
        print(f"Tutolo-2026 in-situ (unfavorable arm) not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**proton-gradient-profile**


In [ ]:
# proton-gradient-profile
show_viz(_render_one('local:ProtonGradientProfile', {'composite': 'viva_genesis.composites.carbon_fixation.hydrothermal_carbon_fixation', 'n_steps': 80, 'title': 'Campaign I — proton gradient → reduced carbon'}, RUNS_DB, STUDY_YAML, overrides=(ov_viva_genesis_composites_carbon_fixation_hydrothermal_carbon_fixation if 'ov_viva_genesis_composites_carbon_fixation_hydrothermal_carbon_fixation' in dir() else None)))

**carbon-fixation-timeseries**


In [ ]:
# carbon-fixation-timeseries
show_viz(_render_one('local:CarbonFixationViz', {'composite': 'viva_genesis.composites.carbon_fixation.hydrothermal_carbon_fixation', 'n_steps': 120, 'title': 'Campaign I — geology to reduced carbon (eQuilibrator + FiPy)'}, RUNS_DB, STUDY_YAML, overrides=(ov_viva_genesis_composites_carbon_fixation_hydrothermal_carbon_fixation if 'ov_viva_genesis_composites_carbon_fixation_hydrothermal_carbon_fixation' in dir() else None)))

**carbon-fixation-phase-map**


In [ ]:
# carbon-fixation-phase-map
show_viz(_render_one('local:CarbonFixationPhaseMap', {'title': 'Campaign I — the carbon-fixation landscape (Decision view)', 'grid_json': 'workspace/studies/og-01-geology-to-reduced-carbon/artifacts/carbon_fixation_phase_grid.json'}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| formate-yield-reproduces-preiner _(primary)_ | kind=derived path=settled_formate_mM detail=batch formate yield mM=None | op range low 50.0 high 220.0 detail formate in Preiner range (~200 mM ceiling) |
| acetate-yield-reproduces-preiner _(primary)_ | kind=derived path=settled_acetate_uM detail=batch acetate yield uM=None | op range low 20.0 high 150.0 detail acetate in Preiner range |
| pyruvate-yield-reproduces-preiner _(primary)_ | kind=derived path=settled_pyruvate_uM detail=batch pyruvate yield uM=None | op range low 2.0 high 15.0 detail pyruvate in Preiner range |
| net-carbon-flux-nonzero-region _(primary)_ | kind=derived path=peak_net_carbon_flux detail=peak organic-C fixation rate | op > value 1e-06 detail peak net carbon flux > 0 (viable region) |
| free-energy-directionality-logged _(primary)_ | kind=derived path=free_energy_logged_fraction detail=fraction of steps with logged ΔG' | op >= value 1.0 detail ΔG' logged every step |
| chemiosmotic-backpressure-loads-gradient _(primary)_ | kind=derived path=pmf_backpressure_sag_frac detail=peak fractional pmf sag while fixation draws the proton reservoir down | op range low 0.05 high 0.95 detail a meaningful back-pressure sag (gradient is spent) without total collapse |
| nernst-planck-flux-conserved _(diagnostic)_ | kind=derived path=max_abs_electro_flux_residual detail=max |FVM residual| | op <= value 0.01 detail max |flux divergence residual| small |
| ph-gradient-drives-formate _(primary)_ | kind=cross_run detail=favorable formate vs no-gradient / no-H2 / no-catalyst | op controls_absent detail formate present with gradient <1 mM in every control None |
| in-situ-ph-correction _(diagnostic)_ | kind=cross_run detail=favorable pmf vs Tutolo-2026 pmf | op arms_reported detail report both arms; flag if the Tutolo pmf collapses |


### Literature anchors

| expectation | model observable | source | status |
| --- | --- | --- | --- |
| Formate up to ~200 mM, acetate ~100 uM, pyruvate ~10 uM at 100 C | product concentrations by mineral/T/P | Preiner 2020 | Met by the favorable arm — batch formate ~200 mM, acetate ~100 uM, pyruvate ~10 uM (Preiner hierarchy), via real eQuilibrator-gated, lab-calibrated mineral kinetics. |
| CO2+H2->formate across pH gradient at room T, ~1.5 bar | formate yield vs delta-pH + controls | Hudson 2020 | Met — formate is present with the pH-gradient proton-motive force and collapses to <1 mM in every control (no gradient / no H2 / no catalyst). |


## Study: Reduced carbon to protometabolism (Campaign II) (`og-02-reduced-carbon-to-protometabolism`)

**Question.** Under continuous feed and washout, do reaction subnetworks become self-reinforcing — persistent cycles, flux amplification, product reuse, autocatalytic motifs — without imposed product accumulation?

**Objective.** Insert an experimentally-motivated reaction graph and test whether metabolism-like subnetworks emerge under flow.

**Hypothesis.** A curated graph (formate/acetate/pyruvate/glyoxylate + rTCA fragments) seeded with Muchowska/Moran edges sustains nonzero throughput and at least one autocatalytic motif within a feasible resource budget.

**Claim.** A curated rTCA-fragment network under continuous feed + washout sustains nonzero carbon throughput, exhibits an autocatalytic motif (OAA self-amplification, gain ~3), and forms a single connected reaction network — metabolism-like behavior emerging without imposed product accumulation.


### Design — originally proposed

_The mechanisms and methods proposed when this study was designed; the model **actually built** (and the tools it uses) is shown below. These may differ as the design was realized._

**Proposed mechanisms**

- E: protometabolic network formation
- D: surface chemistry

**Proposed models / methods**

- SBML + libRoadRunner/Tellurium or COPASI
- SciPy solve_ivp
- element-balance + thermodynamic-consistency checks before kinetic fitting
- GillesPy2 for low-copy stochastic regimes

**Key assumptions**

- Rate laws/mineral dependencies carry provenance; autocatalysis is an emergent observable, not an assumption.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `Reduced carbon to protometabolism (Campaign II)` | `viva_genesis.composites.protometabolism.protometabolic_network` | 0 | — |

Declared parameter sets (`study.yaml` variants):

- **control-no-co2** — `feed_CO2=1e-12`


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `viva_genesis.composites.protometabolism.protometabolic_network`**

A reusable protometabolic reaction-network module. libRoadRunner/Tellurium SBML rTCA-fragment CSTR (pyruvate carboxylation + lumped autocatalytic rTCA turn + reductive branch) fed the og-01 reduced-carbon products under continuous feed+washout. Emergent: sustained carbon throughput, an autocatalytic motif (gain>1), and a connected reaction network. Requires tellurium + networkx.


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_viva_genesis_composites_protometabolism_protometabolic_network = {
    'feed_pyruvate': 0.0001,   # Pyruvate feed set-point (M) from og-01 reduced carbon
    'feed_CO2'     : 0.002,   # CO2 feed set-point (M); ~0 = no-carbon control
    'k_rtca'       : 800.0,   # Autocatalytic rTCA-turn rate constant (Fe-promoted)
    'k_wash'       : 0.05,   # CSTR washout / dilution rate (1/s); the feed-vs-washout axis of the gain phase map
}

### Typed contract (declared interface)

_The bigraph-schema typed ports + conservation predicates this study's composite must satisfy (audited in the workbench §Contract panel). Declared design — independent of whether a run exists yet._

**Inputs**

| port | type | units | bounds |
| --- | --- | --- | --- |
| `formate` | float | molar | >= 0.0 |
| `acetate` | float | molar | >= 0.0 |
| `pyruvate` | float | molar | >= 0.0 |
| `temperature` | float | kelvin | [273.0, 673.0] |
| `pH` | float | dimensionless | [0.0, 14.0] |

**Outputs**

| port | type | units | bounds |
| --- | --- | --- | --- |
| `net_carbon_flux` | float | mole/second |  |

**Conditions**

- `mass_balance` (invariant): `abs(residual_C)+abs(residual_H)+abs(residual_O) < tol`
- `energy_directionality` (invariant): `free_energy_flux_logged == true`
- `reproducibility` (invariant): `solver and db and seed and scenario recorded`


### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: og-02-reduced-carbon-to-protometabolism ===
STUDY = 'og-02-reduced-carbon-to-protometabolism'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_Reduced_carbon_to_protometabolism_Campaign_II = 100
INTERVAL_Reduced_carbon_to_protometabolism_Campaign_II = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_genesis_composites_protometabolism_protometabolic_network if 'ov_viva_genesis_composites_protometabolism_protometabolic_network' in dir() else None  # edits from the panel cell, if any
            spec_viva_genesis_composites_protometabolism_protometabolic_network = materialize_composite('viva_genesis.composites.protometabolism.protometabolic_network', _ov)
            _comp = Composite({'state': spec_viva_genesis_composites_protometabolism_protometabolic_network['state']}, core=core)
            _comp.run(STEPS_Reduced_carbon_to_protometabolism_Campaign_II)
        print('ran', 'Reduced carbon to protometabolism (Campaign II)', f'(viva_genesis.composites.protometabolism.protometabolic_network) for', STEPS_Reduced_carbon_to_protometabolism_Campaign_II, 'steps')
    except Exception as _e:
        print(f"Reduced carbon to protometabolism (Campaign II) not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**protometabolic-network**


In [ ]:
# protometabolic-network
show_viz(_render_one('local:ProtometabolicNetwork', {'composite': 'viva_genesis.composites.protometabolism.protometabolic_network', 'n_steps': 60, 'title': 'Campaign II — the protometabolic network igniting'}, RUNS_DB, STUDY_YAML, overrides=(ov_viva_genesis_composites_protometabolism_protometabolic_network if 'ov_viva_genesis_composites_protometabolism_protometabolic_network' in dir() else None)))

**protometabolism-network**


In [ ]:
# protometabolism-network
show_viz(_render_one('local:ProtometabolismViz', {'composite': 'viva_genesis.composites.protometabolism.protometabolic_network', 'n_steps': 120, 'title': 'Campaign II — protometabolic network (libRoadRunner)'}, RUNS_DB, STUDY_YAML, overrides=(ov_viva_genesis_composites_protometabolism_protometabolic_network if 'ov_viva_genesis_composites_protometabolism_protometabolic_network' in dir() else None)))

**protometabolism-gain-phase-map**


In [ ]:
# protometabolism-gain-phase-map
show_viz(_render_one('local:ProtometabolismGainPhaseMap', {'title': 'Campaign II — the protometabolic ignition landscape (Decision view)', 'grid_json': 'workspace/studies/og-02-reduced-carbon-to-protometabolism/artifacts/protometabolism_gain_grid.json'}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| sustained-carbon-throughput _(primary)_ | kind=derived path=settled_net_carbon_flux detail=steady net carbon fixation rate | op > value 1e-06 detail net throughput > 0 (no imposed accumulation) |
| autocatalytic-gain-present _(primary)_ | kind=derived path=peak_autocatalytic_gain detail=peak OAA self-amplification ratio | op >= value 1.0 detail >= 1 reinforcing/autocatalytic motif |
| network-closure-vs-oneoff _(supporting)_ | kind=derived path=network_connected_final detail=networkx connectivity of the species-reaction graph | op >= value 1.0 detail connected network (not one-offs) |


### Literature anchors

| expectation | model observable | source | status |
| --- | --- | --- | --- |
| Fe2+ promotes 9/11 TCA intermediates + 5 universal precursors from pyruvate/glyoxylate | edge yields + network reachability | Muchowska 2019 | Not yet measurable — module processes not built |


## Study: Dilution to localized individuals (Campaign III) (`og-03-dilution-to-localized-individuals`)

**Question.** When do retention time and concentration cross the thresholds for network persistence, as pore morphology evolves via precipitation/dissolution?

**Objective.** Add pore geometry, adsorption, convection and thermophoresis; find where retention + concentration cross persistence thresholds.

**Hypothesis.** Thermophoresis + convection + adsorption in evolving pores raise local concentration enough that synthesis outruns hydrolysis/leakage in a bounded geometry region; elsewhere washout dominates.

**Claim.** A real FiPy thermal trap (thermophoresis + convection) concentrates a dilute vent feed >10^8-fold in a favorable pore (Baaske 2007), and the concentration collapses to ~1 in an open pore — the geometric threshold at which a localized "individual" forms.


### Design — originally proposed

_The mechanisms and methods proposed when this study was designed; the model **actually built** (and the tools it uses) is shown below. These may differ as the design was realized._

**Proposed mechanisms**

- F: transport/concentration/retention
- B: evolving pores

**Proposed models / methods**

- FiPy finite-volume with Soret term
- OpenPNM/PoreSpy pore geometry
- reproduce Baaske enrichment
- Damkohler/Peclet/Soret scaling

**Key assumptions**

- Concentration arises from the balance of advection, diffusion, thermodiffusion, adsorption, precipitation and degradation — not assumed.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `Dilution to localized individuals (Campaign III)` | `viva_genesis.composites.thermophoretic_concentration.thermophoretic_concentration` | 0 | flow_velocity=0.0001, porosity=0.3, pore_length=0.005, delta_T=30.0 |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `viva_genesis.composites.thermophoretic_concentration.thermophoretic_concentration`**

A reusable thermophoretic-concentration module for the Baaske thermal trap. Module F = genuine FiPy finite-volume accumulation column: thermophoresis (Soret) + buoyant convection concentrate a dilute feed, enrichment exponential in pore length (>10^8 favorable, ~1 in controls). flow_velocity=0 / porosity=1 give the no-trap controls. Requires fipy.


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_viva_genesis_composites_thermophoretic_concentration_thermophoretic_concentration = {
    'flow_velocity': 0.0001,   # Buoyant convective flow in the pore (m/s); 0 = no-convection control
    'porosity'     : 0.3,   # Pore porosity/openness (0-1); ~1 = open-pore (wash-through) control
    'temperature'  : 333.15,   # Mean pore temperature (K)
    'pore_length'  : 0.005,   # Pore length (m); longer = more enrichment
    'delta_T'      : 30.0,   # Cross-pore temperature difference (K)
}

### Typed contract (declared interface)

_The bigraph-schema typed ports + conservation predicates this study's composite must satisfy (audited in the workbench §Contract panel). Declared design — independent of whether a run exists yet._

**Inputs**

| port | type | units | bounds |
| --- | --- | --- | --- |
| `temperature` | float | kelvin | [273.0, 673.0] |
| `flow_velocity` | float | meter/second | >= 0.0 |
| `porosity` | float | dimensionless | [0.0, 1.0] |

**Outputs**

| port | type | units | bounds |
| --- | --- | --- | --- |
| `enrichment_factor` | float | dimensionless | >= 1.0 |
| `residence_time` | float | second | >= 0.0 |

**Conditions**

- `spatial_flux_conservation` (invariant): `divergence(flux) == source_minus_sink`
- `mass_balance` (invariant): `abs(residual_C)+abs(residual_H)+abs(residual_O) < tol`


### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: og-03-dilution-to-localized-individuals ===
STUDY = 'og-03-dilution-to-localized-individuals'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_Dilution_to_localized_individuals_Campaign_III = 100
INTERVAL_Dilution_to_localized_individuals_Campaign_III = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_genesis_composites_thermophoretic_concentration_thermophoretic_concentration if 'ov_viva_genesis_composites_thermophoretic_concentration_thermophoretic_concentration' in dir() else None  # edits from the panel cell, if any
            spec_viva_genesis_composites_thermophoretic_concentration_thermophoretic_concentration = materialize_composite('viva_genesis.composites.thermophoretic_concentration.thermophoretic_concentration', _ov)
            _comp = Composite({'state': spec_viva_genesis_composites_thermophoretic_concentration_thermophoretic_concentration['state']}, core=core)
            _comp.run(STEPS_Dilution_to_localized_individuals_Campaign_III)
        print('ran', 'Dilution to localized individuals (Campaign III)', f'(viva_genesis.composites.thermophoretic_concentration.thermophoretic_concentration) for', STEPS_Dilution_to_localized_individuals_Campaign_III, 'steps')
    except Exception as _e:
        print(f"Dilution to localized individuals (Campaign III) not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**thermophoretic-trap**


In [ ]:
# thermophoretic-trap
show_viz(_render_one('local:ThermophoreticTrapViz', {'composite': 'viva_genesis.composites.thermophoretic_concentration.thermophoretic_concentration', 'title': 'Campaign III — the thermal trap concentrates an individual'}, RUNS_DB, STUDY_YAML, overrides=(ov_viva_genesis_composites_thermophoretic_concentration_thermophoretic_concentration if 'ov_viva_genesis_composites_thermophoretic_concentration_thermophoretic_concentration' in dir() else None)))

**pore-ignition-coupling**


In [ ]:
# pore-ignition-coupling
show_viz(_render_one('local:PoreIgnitionCoupling', {'title': 'Campaign III — localization makes the individual (Decision view)', 'grid_json': 'workspace/studies/og-02-reduced-carbon-to-protometabolism/artifacts/protometabolism_gain_grid.json'}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| thermophoretic-enrichment-reproduces-baaske _(primary)_ | kind=derived path=settled_accumulation_peclet detail=ln(enrichment) = beta × pore length | op >= value 18.0 detail Pe_acc ≥ ln(1e8) ≈ 18.4 in favorable geometry |
| retention-beats-destruction _(primary)_ | kind=derived path=log10_enrichment detail=log10 of the settled enrichment fold | op >= value 6.0 detail ≥10^6-fold boost (synthesis outruns dilution) |
| open-pore-control-does-not-concentrate _(primary)_ | kind=cross_run detail=favorable enrichment vs open-pore control | op controls_absent detail enrichment collapses to ~1 in the open-pore control |
| enrichment-exponential-in-pore-length _(diagnostic)_ | kind=cross_run detail=favorable vs 10× shorter pore | op arms_reported detail short pore enriches ≥100× less than the favorable pore |
| spatial-flux-conserved _(diagnostic)_ | kind=derived path=max_abs_trap_flux_residual detail=max |FVM flux-divergence residual| | op <= value 0.01 detail FVM residual small (direct LU solve) |


### Literature anchors

| expectation | model observable | source | status |
| --- | --- | --- | --- |
| >10^8-fold nucleotide accumulation in favorable modeled pore geometries | enrichment vs geometry | Baaske 2007 | Not yet measurable — module processes not built |


## Study: Mineral compartments to protocells (Campaign IV) (`og-04-mineral-compartments-to-protocells`)

**Question.** Do mixed-amphiphile organic boundaries enhance persistence without cutting the system off from geochemical energy?

**Objective.** Introduce mixed amphiphiles + membrane assembly; measure whether organic boundaries enhance persistence without isolating the system.

**Hypothesis.** Over a salt/pH/divalent-cation/T window, vesicles form and selectively retain while remaining permeable enough to exploit the external gradient — a non-empty individuality-vs-access region exists.

**Claim.** Mixed single-chain amphiphiles form viable vesicles across the alkaline / seawater-salt window where single fatty acids fail (Jordan 2019), and those compartments retain internal chemistry while staying permeable to the gradient — a non-empty individuality-without-isolation region.


### Design — originally proposed

_The mechanisms and methods proposed when this study was designed; the model **actually built** (and the tools it uses) is shown below. These may differ as the design was realized._

**Proposed mechanisms**

- G: amphiphiles, mineral pores and protocells

**Proposed models / methods**

- custom coarse-grained membrane kinetics
- Smoldyn/ReaDDy spatial stochastic
- OpenMM/LAMMPS to calibrate permeability laws

**Key assumptions**

- Mineral pores provide initial individuality; organic boundaries are a later, condition-dependent transition.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `Mineral compartments to protocells (Campaign IV)` | `viva_genesis.composites.mixed_amphiphile_protocell.mixed_amphiphile_protocell` | 0 | mix_fraction=0.4, pH=9.0, ionic_strength=0.35, temperature=318.15 |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `viva_genesis.composites.mixed_amphiphile_protocell.mixed_amphiphile_protocell`**

A reusable mixed-amphiphile protocell module (Module G). Coarse-grained vesicle self-assembly + permeability calibrated to Jordan 2019: mixed single-chain amphiphiles broaden the pH / salt / divalent stability window and form vesicles that retain chemistry while staying permeable to the gradient (individuality without isolation). mix_fraction≈0 / high salt / acidic pH give the controls. Coarse-grained reproduction (not wrapped MD).


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_viva_genesis_composites_mixed_amphiphile_protocell_mixed_amphiphile_protocell = {
    'mix_fraction'  : 0.4,   # Co-surfactant fraction (0 = single fatty acid; higher = broader Jordan window)
    'pH'            : 9.0,   # Environment pH (alkaline vent)
    'ionic_strength': 0.35,   # Ionic strength (M); ~seawater 0.35
    'temperature'   : 318.15,   # Temperature (K)
}

### Typed contract (declared interface)

_The bigraph-schema typed ports + conservation predicates this study's composite must satisfy (audited in the workbench §Contract panel). Declared design — independent of whether a run exists yet._

**Inputs**

| port | type | units | bounds |
| --- | --- | --- | --- |
| `pH` | float | dimensionless | [0.0, 14.0] |
| `ionic_strength` | float | molar | >= 0.0 |
| `temperature` | float | kelvin | [273.0, 673.0] |

**Outputs**

| port | type | units | bounds |
| --- | --- | --- | --- |
| `membrane_area` | float | meter**2 | >= 0.0 |
| `permeability` | float | meter/second | >= 0.0 |
| `compartment_volume` | float | liter | >= 0.0 |
| `viable_compartments` | integer | dimensionless | >= 0 |

**Conditions**

- `mass_balance` (invariant): `abs(residual_C)+abs(residual_H)+abs(residual_O) < tol`
- `spatial_flux_conservation` (invariant): `divergence(flux) == source_minus_sink`
- `energy_directionality` (invariant): `free_energy_flux_logged == true`


### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: og-04-mineral-compartments-to-protocells ===
STUDY = 'og-04-mineral-compartments-to-protocells'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_Mineral_compartments_to_protocells_Campaign_IV = 100
INTERVAL_Mineral_compartments_to_protocells_Campaign_IV = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_genesis_composites_mixed_amphiphile_protocell_mixed_amphiphile_protocell if 'ov_viva_genesis_composites_mixed_amphiphile_protocell_mixed_amphiphile_protocell' in dir() else None  # edits from the panel cell, if any
            spec_viva_genesis_composites_mixed_amphiphile_protocell_mixed_amphiphile_protocell = materialize_composite('viva_genesis.composites.mixed_amphiphile_protocell.mixed_amphiphile_protocell', _ov)
            _comp = Composite({'state': spec_viva_genesis_composites_mixed_amphiphile_protocell_mixed_amphiphile_protocell['state']}, core=core)
            _comp.run(STEPS_Mineral_compartments_to_protocells_Campaign_IV)
        print('ran', 'Mineral compartments to protocells (Campaign IV)', f'(viva_genesis.composites.mixed_amphiphile_protocell.mixed_amphiphile_protocell) for', STEPS_Mineral_compartments_to_protocells_Campaign_IV, 'steps')
    except Exception as _e:
        print(f"Mineral compartments to protocells (Campaign IV) not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**protocell-window**


In [ ]:
# protocell-window
show_viz(_render_one('local:ProtocellWindowViz', {'composite': 'viva_genesis.composites.mixed_amphiphile_protocell.mixed_amphiphile_protocell', 'title': 'Campaign IV — the protocell window broadens (mixed amphiphiles)'}, RUNS_DB, STUDY_YAML, overrides=(ov_viva_genesis_composites_mixed_amphiphile_protocell_mixed_amphiphile_protocell if 'ov_viva_genesis_composites_mixed_amphiphile_protocell_mixed_amphiphile_protocell' in dir() else None)))

**protocell-retention-access**


In [ ]:
# protocell-retention-access
show_viz(_render_one('local:ProtocellPhaseMap', {'title': 'Campaign IV — a boundary that retains without isolating (Decision view)'}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| vesicles-form-in-window _(primary)_ | kind=derived path=settled_viable_compartments detail=viable vesicle count in the favorable arm | op > value 0.0 detail vesicles form (>0) in the mixed-amphiphile window |
| retention-without-isolation _(primary)_ | kind=derived path=settled_individuality_access detail=retention × access | op >= value 0.3 detail both retention and access nonzero (a non-empty individuality-vs-access region) |
| mixed-amphiphile-advantage-over-single _(primary)_ | kind=cross_run detail=favorable (mixed) viable vs single-fatty-acid control | op controls_absent detail mixed forms vesicles; single fatty acid does not at the same salt |
| membrane-transition-robust _(diagnostic)_ | kind=cross_run detail=favorable vs extreme-salt and acidic controls | op arms_reported detail extreme-salt and acidic controls abolish vesicles (survival region is bounded) |


### Literature anchors

| expectation | model observable | source | status |
| --- | --- | --- | --- |
| Mixed single-chain amphiphiles form vesicles across broader alkaline pH/salinity/divalent/T than single-fatty-acid models | vesicle stability map | Jordan 2019 | Not yet measurable — module processes not built |


## Study: Environmental to internal bioenergetics (Campaign V) (`og-05-environmental-to-internal-bioenergetics`)

**Question.** Can selection move from externally imposed gradients toward internally maintained electrochemical states (catalytic ion transport then pumping)?

**Objective.** Allow catalytic ion transport and active pumping; test whether selection favors internalized gradient generation.

**Hypothesis.** When membrane permeability, Na+/H+ coupling and catalyst efficiency vary under selection, systems with increasing internal gradient generation win on throughput + persistence — but only if membranes stay sufficiently leaky (Sojo constraint).

**Claim.** Under a declining external gradient, selection drives an efficient-pump population to internalize gradient generation (pump fraction 0→0.95) and persist, while a weak-pump lineage collapses — the Sojo-2014 leaky-membrane constraint on bioenergetic internalization.


### Design — originally proposed

_The mechanisms and methods proposed when this study was designed; the model **actually built** (and the tools it uses) is shown below. These may differ as the design was realized._

**Proposed mechanisms**

- H: bioenergetic internalization

**Proposed models / methods**

- evolutionary control loop over membrane parameters
- reward = throughput + persistence
- Sojo leaky-membrane model as baseline

**Key assumptions**

- Internalization is an evolutionary control problem, not a scripted upgrade.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `Environmental to internal bioenergetics (Campaign V)` | `viva_genesis.composites.bioenergetic_internalization.bioenergetic_internalization` | 0 | catalyst_efficiency=1.0, v_start=0.4, decline_rate=0.01 |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `viva_genesis.composites.bioenergetic_internalization.bioenergetic_internalization`**

A reusable bioenergetic-internalization module (Module H). An evolutionary control loop (replicator dynamics, Sojo-2014 leaky-membrane fitness) under a declining external gradient: as the geochemical pmf collapses, an efficient-pump population internalizes gradient generation and persists, while a weak-pump control collapses (the Sojo constraint). catalyst_efficiency sets the arm. Coarse-grained, deterministic reproduction (not wrapped ALife).


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_viva_genesis_composites_bioenergetic_internalization_bioenergetic_internalization = {
    'catalyst_efficiency': 1.0,   # Active-pump (Na+/H+) efficiency (1.0 efficient; ~0.3 weak-pump control)
    'v_start'            : 0.4,   # Initial external gradient (V)
    'decline_rate'       : 0.01,   # External-gradient decline per step (V/s)
}

### Typed contract (declared interface)

_The bigraph-schema typed ports + conservation predicates this study's composite must satisfy (audited in the workbench §Contract panel). Declared design — independent of whether a run exists yet._

**Inputs**

| port | type | units | bounds |
| --- | --- | --- | --- |
| `electric_potential` | float | volt |  |
| `catalyst_efficiency` | float | dimensionless | >= 0.0 |

**Outputs**

| port | type | units | bounds |
| --- | --- | --- | --- |
| `lineage_fitness` | float | dimensionless |  |
| `free_energy_flux` | float | watt |  |
| `internalization` | float | dimensionless | [0.0, 1.0] |
| `survivors` | float | dimensionless | [0.0, 1.0] |
| `mean_permeability` | float | meter/second | >= 0.0 |

**Conditions**

- `energy_directionality` (invariant): `free_energy_flux_logged == true`
- `charge_neutrality` (invariant): `abs(net_charge) < tol`
- `reproducibility` (invariant): `solver and db and seed and scenario recorded`


### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: og-05-environmental-to-internal-bioenergetics ===
STUDY = 'og-05-environmental-to-internal-bioenergetics'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_Environmental_to_internal_bioenergetics_Campaign_V = 100
INTERVAL_Environmental_to_internal_bioenergetics_Campaign_V = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_genesis_composites_bioenergetic_internalization_bioenergetic_internalization if 'ov_viva_genesis_composites_bioenergetic_internalization_bioenergetic_internalization' in dir() else None  # edits from the panel cell, if any
            spec_viva_genesis_composites_bioenergetic_internalization_bioenergetic_internalization = materialize_composite('viva_genesis.composites.bioenergetic_internalization.bioenergetic_internalization', _ov)
            _comp = Composite({'state': spec_viva_genesis_composites_bioenergetic_internalization_bioenergetic_internalization['state']}, core=core)
            _comp.run(STEPS_Environmental_to_internal_bioenergetics_Campaign_V)
        print('ran', 'Environmental to internal bioenergetics (Campaign V)', f'(viva_genesis.composites.bioenergetic_internalization.bioenergetic_internalization) for', STEPS_Environmental_to_internal_bioenergetics_Campaign_V, 'steps')
    except Exception as _e:
        print(f"Environmental to internal bioenergetics (Campaign V) not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**bioenergetic-internalization**


In [ ]:
# bioenergetic-internalization
show_viz(_render_one('local:BioenergeticInternalizationViz', {'composite': 'viva_genesis.composites.bioenergetic_internalization.bioenergetic_internalization', 'title': 'Campaign V — selection internalizes the gradient'}, RUNS_DB, STUDY_YAML, overrides=(ov_viva_genesis_composites_bioenergetic_internalization_bioenergetic_internalization if 'ov_viva_genesis_composites_bioenergetic_internalization_bioenergetic_internalization' in dir() else None)))

**bioenergetic-provenance**


In [ ]:
# bioenergetic-provenance
show_viz(_render_one('local:BioenergeticProvenanceViz', {'title': 'Campaign V — the organism begins to own its energy (Decision view)'}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| selection-internalizes-gradient-generation _(primary)_ | kind=derived path=final_internalization detail=active-pump fraction of the surviving population at the collapsed gradient | op >= value 0.6 detail the population internalizes (pump fraction >= 0.6) as the external gradient falls |
| reduced-gradient-dependence _(primary)_ | kind=derived path=final_survivors detail=surviving strategy mass at the collapsed external gradient | op >= value 0.5 detail internalizing lineage persists (survivors >= 0.5) at the collapsed gradient |
| sojo-constraint-weak-pump-collapses _(primary)_ | kind=cross_run detail=efficient-pump survivors vs weak-pump control | op controls_absent detail the weak-pump control collapses (survivors ~0) where the efficient pump persists |
| energy-cost-accounted _(supporting)_ | kind=derived path=final_internalization detail=internalized population has net-positive flux | op > value 0.0 detail a net-positive-flux internalized region exists |


### Literature anchors

| expectation | model observable | source | status |
| --- | --- | --- | --- |
| LUCA-like cells can use proton gradients only if membranes remain sufficiently leaky | permeability vs viability | Sojo 2014 / Lane 2012 | Not yet measurable — module processes not built |


## Study: Chemistry to heredity (Campaign VI) (`og-06-chemistry-to-heredity`)

**Question.** Does inherited state improve persistence/throughput — comparing RNA-like, peptide-assisted, compositional and hybrid heredity — rather than template replication in isolation? (Do NOT presume RNA.)

**Objective.** Introduce competing heredity modules and test whether inherited state improves persistence under feasible budgets.

**Hypothesis.** At least one heredity module raises lineage persistence + information retention above its non-heritable control under a feasible resource budget; the winner is not assumed a priori.

**Claim.** An inherited chemical state improves lineage fitness above its replication cost when it clears the Eigen error threshold — but heredity need NOT presume a long high-fidelity RNA template. An agent-based lineage shows template heredity suffers the length catastrophe (Q=fidelity^length collapses with length) while compositional (GARD) and template-focused hybrid heredity persist length-independently (functional persistence ~0.88, seed-robust across 8 seeds) above the non-heritable control, which loses the parent↔offspring correlation and collapses. (A mean-field quasispecies arm still ranks a long high-fidelity template above a cheaper peptide module above non-inheritance when all clear the threshold.)


### Design — originally proposed

_The mechanisms and methods proposed when this study was designed; the model **actually built** (and the tools it uses) is shown below. These may differ as the design was realized._

**Proposed mechanisms**

- J: polymers, information and heredity (as competing modules)

**Proposed models / methods**

- GillesPy2/StochPy SSA
- rule-based chemistry (BioNetGen/NFsim) for sequence combinatorics
- separate catalyst discovery from inheritance
- explicit fidelity/cost/competition

**Key assumptions**

- Treat informational chemistry as a branch point; do not presume a viable RNA replication mechanism.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `Chemistry to heredity (Campaign VI)` | `viva_genesis.composites.heredity_module_comparison.heredity_module_comparison` | 0 | module=rna-like, temperature=353.15 |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `viva_genesis.composites.heredity_module_comparison.heredity_module_comparison`**

A reusable heredity-module comparison (Module J). Genuine libRoadRunner quasispecies replicator (Eigen error threshold): a heredity module (RNA-like / peptide / low-fidelity / non-heritable) replicates a polymer with whole-sequence fidelity Q = fidelity^length; above the threshold the functional information is retained and inheritance pays, below it errors win. Compares modules head-to-head on lineage fitness. Requires tellurium/libroadrunner.


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_viva_genesis_composites_heredity_module_comparison_heredity_module_comparison = {
    'module'     : 'rna-like',   # Heredity-module preset: rna-like | peptide | low-fidelity | non-heritable
    'temperature': 353.15,   # Temperature (K; ~80 C vent)
}

### Typed contract (declared interface)

_The bigraph-schema typed ports + conservation predicates this study's composite must satisfy (audited in the workbench §Contract panel). Declared design — independent of whether a run exists yet._

**Inputs**

| port | type | units | bounds |
| --- | --- | --- | --- |
| `temperature` | float | kelvin | [273.0, 673.0] |

**Outputs**

| port | type | units | bounds |
| --- | --- | --- | --- |
| `polymer_length` | integer | dimensionless | >= 1 |
| `lineage_fitness` | float | dimensionless |  |
| `information_retention` | float | dimensionless | [0.0, 1.0] |
| `functional_templates` | float | dimensionless | >= 0.0 |

**Conditions**

- `mass_balance` (invariant): `abs(residual_C)+abs(residual_H)+abs(residual_O) < tol`
- `reproducibility` (invariant): `solver and db and seed and scenario recorded`
- `energy_directionality` (invariant): `free_energy_flux_logged == true`


### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: og-06-chemistry-to-heredity ===
STUDY = 'og-06-chemistry-to-heredity'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_Chemistry_to_heredity_Campaign_VI = 100
INTERVAL_Chemistry_to_heredity_Campaign_VI = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_genesis_composites_heredity_module_comparison_heredity_module_comparison if 'ov_viva_genesis_composites_heredity_module_comparison_heredity_module_comparison' in dir() else None  # edits from the panel cell, if any
            spec_viva_genesis_composites_heredity_module_comparison_heredity_module_comparison = materialize_composite('viva_genesis.composites.heredity_module_comparison.heredity_module_comparison', _ov)
            _comp = Composite({'state': spec_viva_genesis_composites_heredity_module_comparison_heredity_module_comparison['state']}, core=core)
            _comp.run(STEPS_Chemistry_to_heredity_Campaign_VI)
        print('ran', 'Chemistry to heredity (Campaign VI)', f'(viva_genesis.composites.heredity_module_comparison.heredity_module_comparison) for', STEPS_Chemistry_to_heredity_Campaign_VI, 'steps')
    except Exception as _e:
        print(f"Chemistry to heredity (Campaign VI) not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**heredity-lineage-tree**


In [ ]:
# heredity-lineage-tree
show_viz(_render_one('local:HeredityLineageTree', {'composite': 'viva_genesis.composites.heredity_lineage.heredity_lineage_comparison', 'grid_json': 'workspace/studies/og-06-chemistry-to-heredity/artifacts/heredity_error_threshold_grid.json', 'title': 'Campaign VI — heredity as an inherited lineage'}, RUNS_DB, STUDY_YAML))

**heredity-error-threshold**


In [ ]:
# heredity-error-threshold
show_viz(_render_one('local:HeredityErrorThresholdViz', {'composite': 'viva_genesis.composites.heredity_module_comparison.heredity_module_comparison', 'title': 'Campaign VI — heredity above the error threshold'}, RUNS_DB, STUDY_YAML, overrides=(ov_viva_genesis_composites_heredity_module_comparison_heredity_module_comparison if 'ov_viva_genesis_composites_heredity_module_comparison_heredity_module_comparison' in dir() else None)))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| inheritance-beats-noninheritance _(primary)_ | kind=derived path=final_lineage_fitness detail=RNA-like lineage fitness at steady state | op > value 0.1 detail inheriting lineage fitness net-positive (and beats the non-heritable control) |
| information-retained-above-error-threshold _(primary)_ | kind=derived path=final_information_retention detail=C/(C+E) retention at steady state | op >= value 0.4 detail functional information retained (above the error threshold) |
| heredity-modules-compared _(primary)_ | kind=cross_run detail=RNA-like vs peptide lineage fitness | op arms_reported detail both modules viable; RNA-like ranks above peptide |
| error-catastrophe-below-threshold _(diagnostic)_ | kind=cross_run detail=RNA-like vs low-fidelity module | op controls_absent detail the low-fidelity module collapses (retention ~0 fitness negative) None |
| replication-feasible-budget _(supporting)_ | kind=derived path=final_lineage_fitness detail=net fitness after replication cost | op > value 0.0 detail net-positive within the replication budget |
| heredity-lineage-compositional-persists _(primary)_ | kind=cross_run detail=compositional vs non-heritable functional persistence in the agent-based lineage | op arms_reported detail compositional persistence above threshold and above the non-heritable control (realized heritability separates them) |


### Literature anchors

| expectation | model observable | source | status |
| --- | --- | --- | --- |
| Mineral/CO-driven peptide formation at 100 C, pH 7-10 | peptide formation rate/conditions | Huber & Wachtershauser 1998 | Not yet measurable — module processes not built |


## Study: Ecology before LUCA (Campaign VII) (`og-07-ecology-before-luca`)

**Question.** In connected pore networks with cross-feeding, parasitism, cooperation and horizontal exchange, what emergent capabilities arise — compared to ancestral reconstructions, without assuming LUCA is primitive?

**Objective.** Run heterogeneous populations over a connected pore network; compare emergent capability redundancy (robustness) to the isolated counterfactual, and the collective repertoire to the LUCA reconstruction as a constraint.

**Hypothesis.** Heterogeneous populations develop metabolic diversity, cross-feeding and robustness single compartments do not; the emergent capability set overlaps the anaerobic H2/CO2/FeS LUCA reconstruction as a boundary.

**Claim.** A connected metacommunity of heterogeneous pore compartments develops functional redundancy (robustness) that an isolated compartment cannot — cross-feeding and horizontal transfer spread coarse-grained protometabolic capabilities into >= 2-fold coverage, so most functions survive a single-compartment loss. Robustness EMERGES only in a bounded region of (cross-feeding exchange_rate x inter-compartment diversity) — too little diversity leaves nothing to share, too little exchange leaves a broad repertoire fragile. The collective repertoire's overlap with the anaerobic H2/CO2/FeS LUCA reconstruction is REPORTED as a far-end boundary the emergent set approaches (~0.6-0.7), not a target — LUCA constrains which capabilities are eventually acquired, not evidence the first compartments were LUCA-like.


### Design — originally proposed

_The mechanisms and methods proposed when this study was designed; the model **actually built** (and the tools it uses) is shown below. These may differ as the design was realized._

**Proposed mechanisms**

- ecology: populations + evolution
- LUCA boundary condition

**Proposed models / methods**

- Vivarium multi-agent composites over shared spatial fields
- COBRApy for late-stage genome-scale benchmarks
- evolutionary algorithms

**Key assumptions**

- Use LUCA as a constraint on capabilities eventually acquired, not evidence the first replicators were LUCA-like.
- Each compartment is a coarse-grained capability vector over a protometabolic function set, not an instantiated metabolic network.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `Ecology before LUCA (Campaign VII)` | `viva_genesis.composites.cross_feeding_metacommunity.cross_feeding_metacommunity` | 0 | n_compartments=8, exchange_rate=0.3, diversity=0.75, hgt_rate=0.1, seed=1 |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `viva_genesis.composites.cross_feeding_metacommunity.cross_feeding_metacommunity`**

A reusable cross-feeding metacommunity module (Module L). N connected compartments, each a coarse-grained capability vector over a protometabolic function set, evolving under cross-feeding (byproduct of one feeds the next capability in a neighbour), horizontal transfer, and parasite/cooperator roles. Connected heterogeneous populations develop capability redundancy (robustness) and a collective repertoire single compartments do not; the emergent set's overlap with the anaerobic H2/CO2/FeS LUCA reconstruction is reported as a far-end boundary, not a target. Coarse-grained, deterministic reproduction (not wrapped ALife).


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_viva_genesis_composites_cross_feeding_metacommunity_cross_feeding_metacommunity = {
    'n_compartments': 8,   # Number of connected compartments
    'exchange_rate' : 0.3,   # Cross-feeding strength (use isolated flag, not 0, to disable)
    'diversity'     : 0.75,   # Inter-compartment heterogeneity (0 identical; 1 all different)
    'parasite_load' : 0.0,   # Fraction of compartments that drain neighbours
    'hgt_rate'      : 0.1,   # Horizontal-transfer rate (0 = no spread)
    'isolated'      : False,   # Isolated counterfactual arm: no cross-feeding/HGT/parasitism (use this, not exchange_rate=0)
    'seed'          : 1,   # RNG seed (reproducible assignment)
}

### Typed contract (declared interface)

_The bigraph-schema typed ports + conservation predicates this study's composite must satisfy (audited in the workbench §Contract panel). Declared design — independent of whether a run exists yet._

**Inputs**

| port | type | units | bounds |
| --- | --- | --- | --- |
| `temperature` | float | kelvin | [273.0, 673.0] |
| `flow_velocity` | float | meter/second | >= 0.0 |

**Outputs**

| port | type | units | bounds |
| --- | --- | --- | --- |
| `lineage_fitness` | float | dimensionless |  |

**Conditions**

- `mass_balance` (invariant): `abs(residual_C)+abs(residual_H)+abs(residual_O) < tol`
- `energy_directionality` (invariant): `free_energy_flux_logged == true`
- `spatial_flux_conservation` (invariant): `divergence(flux) == source_minus_sink`
- `reproducibility` (invariant): `solver and db and seed and scenario recorded`


### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: og-07-ecology-before-luca ===
STUDY = 'og-07-ecology-before-luca'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_Ecology_before_LUCA_Campaign_VII = 100
INTERVAL_Ecology_before_LUCA_Campaign_VII = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_genesis_composites_cross_feeding_metacommunity_cross_feeding_metacommunity if 'ov_viva_genesis_composites_cross_feeding_metacommunity_cross_feeding_metacommunity' in dir() else None  # edits from the panel cell, if any
            spec_viva_genesis_composites_cross_feeding_metacommunity_cross_feeding_metacommunity = materialize_composite('viva_genesis.composites.cross_feeding_metacommunity.cross_feeding_metacommunity', _ov)
            _comp = Composite({'state': spec_viva_genesis_composites_cross_feeding_metacommunity_cross_feeding_metacommunity['state']}, core=core)
            _comp.run(STEPS_Ecology_before_LUCA_Campaign_VII)
        print('ran', 'Ecology before LUCA (Campaign VII)', f'(viva_genesis.composites.cross_feeding_metacommunity.cross_feeding_metacommunity) for', STEPS_Ecology_before_LUCA_Campaign_VII, 'steps')
    except Exception as _e:
        print(f"Ecology before LUCA (Campaign VII) not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**metacommunity-phase-map**


In [ ]:
# metacommunity-phase-map
show_viz(_render_one('local:MetacommunityPhaseMap', {'composite': 'viva_genesis.composites.cross_feeding_metacommunity.cross_feeding_metacommunity', 'grid_json': 'workspace/studies/og-07-ecology-before-luca/artifacts/metacommunity_robustness_grid.json', 'title': 'Campaign VII — where cross-feeding robustness emerges'}, RUNS_DB, STUDY_YAML, overrides=(ov_viva_genesis_composites_cross_feeding_metacommunity_cross_feeding_metacommunity if 'ov_viva_genesis_composites_cross_feeding_metacommunity_cross_feeding_metacommunity' in dir() else None)))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| cross-feeding-robustness _(primary)_ | kind=derived path=metacommunity_robustness detail=settled metacommunity robustness (functional redundancy) of the connected run | op > value 0.5 detail a majority of capabilities are redundantly covered (robust to a single-compartment loss) |
| connected-beats-isolated _(primary)_ | kind=cross_run detail=connected vs isolated metacommunity robustness across an 8-seed sweep | op arms_reported detail connected robustness strictly higher than isolated on every seed (measured min gain reported) |
| luca-capability-overlap _(supporting)_ | kind=cross_run detail=collective repertoire overlap with the anaerobic LUCA core (Weiss 2016) | op arms_reported detail overlap reported as a boundary not a target None |


### Literature anchors

| expectation | model observable | source | status |
| --- | --- | --- | --- |
| LUCA: anaerobic, H2-dependent, CO2-fixing, FeS-rich (355 protein families) | emergent capability overlap | Weiss 2016 / Moody 2024 | Measured — collective repertoire overlaps the anaerobic LUCA core at ~0.6-0.7 (reported boundary) |


## Study: Escape from the vent (Campaign VIII) (`og-08-escape-from-the-vent`)

**Question.** What is the minimum set of innovations required for autonomous persistence away from the geological reactor, as environmental support is relaxed?

**Objective.** Relax environmental support via ablation sweeps and identify the minimum innovation set for autonomy.

**Hypothesis.** A bounded innovation set (internal energy conversion + resource import + heredity) allows persistence as gradient strength/mineral access/feed are reduced; below it, systems stay obligately vent-tied.

**Claim.** Autonomy away from the geological reactor requires an irreducible innovation set — internal energy conversion (a pump), active resource import, and heredity to hold them. Because the lineage's budget is co-limited (it needs BOTH usable free energy AND usable resources, Liebig's minimum), a single innovation is a cost without a co-benefit; only the full set keeps the net budget above the survival floor as the external gradient, mineral access and feed are all withdrawn. A knockout sweep shows the full set persists off-gradient (surviving fraction 1.0) while removing any one innovation collapses persistence to 0 — so the minimum robust set for escaping the vent is {pump, importer, heredity}.


### Design — originally proposed

_The mechanisms and methods proposed when this study was designed; the model **actually built** (and the tools it uses) is shown below. These may differ as the design was realized._

**Proposed mechanisms**

- all modules integrated
- autonomy readout

**Proposed models / methods**

- multi-agent composites with relaxed boundaries
- ablation sweeps over gradient/mineral/feed
- population genetics

**Key assumptions**

- Autonomy is reduced dependence on the exact vent interface, measured by ablation — not a binary.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `Escape from the vent (Campaign VIII)` | `viva_genesis.composites.off_gradient_persistence.off_gradient_persistence` | 0 | — |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `viva_genesis.composites.off_gradient_persistence.off_gradient_persistence`**

A reusable off-gradient-persistence module (Module M). A lineage's co-limited (Liebig-minimum) free-energy + resource budget under relaxing geological support (external gradient, mineral access, feed all declining over the run), with three toggleable innovations: internal energy conversion (has_pump), active resource import (has_importer) and information maintenance (has_heredity). Because the budget is co-limited the innovation set is irreducible — as support -> 0 only the full set keeps net budget above the survival floor, so a knockout sweep identifies {pump, importer, heredity} as the minimum robust set for autonomy. Coarse-grained, deterministic reproduction (not a wrapped ALife engine).


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_viva_genesis_composites_off_gradient_persistence_off_gradient_persistence = {
    'gradient_support'   : 1.0,   # Starting external-gradient support, relaxed to 0 over the run (dimensionless, 0-1)
    'mineral_access'     : 1.0,   # Starting mineral / FeS-catalyst access, relaxed to 0 (dimensionless, 0-1)
    'feed'               : 1.0,   # Starting geochemical feed, relaxed to 0 (dimensionless, 0-1)
    'has_pump'           : 1.0,   # Internal energy conversion innovation (1 present; 0 knockout)
    'has_importer'       : 1.0,   # Active resource-import innovation (1 present; 0 knockout)
    'has_heredity'       : 1.0,   # Information-maintenance innovation; without it the machinery degrades (1 present; 0 knockout)
    'catalyst_efficiency': 1.0,   # Internal-pump catalytic efficiency (dimensionless)
    'decline_rate'       : 0.02,   # Support relaxation per step (1/s); support reaches 0 partway through the run
}

### Typed contract (declared interface)

_The bigraph-schema typed ports + conservation predicates this study's composite must satisfy (audited in the workbench §Contract panel). Declared design — independent of whether a run exists yet._

**Inputs**

| port | type | units | bounds |
| --- | --- | --- | --- |
| `gradient_support` | float | dimensionless | [0.0, 1.0] |
| `mineral_access` | float | dimensionless | [0.0, 1.0] |
| `feed` | float | dimensionless | [0.0, 1.0] |

**Outputs**

| port | type | units | bounds |
| --- | --- | --- | --- |
| `survivors` | float | dimensionless | [0.0, 1.0] |
| `lineage_fitness` | float | dimensionless |  |
| `free_energy_flux` | float | watt |  |

**Conditions**

- `energy_directionality` (invariant): `free_energy_flux >= 0 and co-limited budget (min of energy, resource)`
- `reproducibility` (invariant): `solver and db and seed and scenario recorded`


### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: og-08-escape-from-the-vent ===
STUDY = 'og-08-escape-from-the-vent'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_Escape_from_the_vent_Campaign_VIII = 100
INTERVAL_Escape_from_the_vent_Campaign_VIII = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_genesis_composites_off_gradient_persistence_off_gradient_persistence if 'ov_viva_genesis_composites_off_gradient_persistence_off_gradient_persistence' in dir() else None  # edits from the panel cell, if any
            spec_viva_genesis_composites_off_gradient_persistence_off_gradient_persistence = materialize_composite('viva_genesis.composites.off_gradient_persistence.off_gradient_persistence', _ov)
            _comp = Composite({'state': spec_viva_genesis_composites_off_gradient_persistence_off_gradient_persistence['state']}, core=core)
            _comp.run(STEPS_Escape_from_the_vent_Campaign_VIII)
        print('ran', 'Escape from the vent (Campaign VIII)', f'(viva_genesis.composites.off_gradient_persistence.off_gradient_persistence) for', STEPS_Escape_from_the_vent_Campaign_VIII, 'steps')
    except Exception as _e:
        print(f"Escape from the vent (Campaign VIII) not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**off-gradient-survival-frontier**


In [ ]:
# off-gradient-survival-frontier
show_viz(_render_one('local:OffGradientPhaseMap', {'composite': 'viva_genesis.composites.off_gradient_persistence.off_gradient_persistence', 'grid_json': 'workspace/studies/og-08-escape-from-the-vent/artifacts/off_gradient_grid.json', 'title': 'Campaign VIII — the survival frontier off the gradient'}, RUNS_DB, STUDY_YAML, overrides=(ov_viva_genesis_composites_off_gradient_persistence_off_gradient_persistence if 'ov_viva_genesis_composites_off_gradient_persistence_off_gradient_persistence' in dir() else None)))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| survival-off-gradient _(primary)_ | kind=derived path=offgradient_survival detail=surviving fraction at the end of the run as support -> 0=None | op >= value 0.5 detail a lineage persists off-gradient (surviving fraction >= 0.5) |
| minimum-innovation-set _(primary)_ | kind=cross_run detail=full set vs each single knockout surviving fraction off-gradient=None | op arms_reported detail full set persists off-gradient and every single knockout collapses (each innovation is necessary) |


### Literature anchors

| expectation | model observable | source | status |
| --- | --- | --- | --- |
| Alkaline-vent origin — the first cells are vent-tied, dependent on the geochemical proton gradient and FeS mineral catalysis; escaping to free-living requires internalising energy generation. | off-gradient persistence vs innovation set | Martin & Russell 2007; Lane & Martin 2012; Sojo et al. 2014 | Reproduced as a coarse-grained co-limited budget — the full innovation set persists off-gradient; each single knockout collapses. |


## Study: Adversarial: generic nonequilibrium control (`og-adv-generic-nonequilibrium`)

**Question.** Does a generic nonequilibrium chemical system with no vent-specific features (no sustained H2, no Fe/Ni/S catalytic interface, no pore concentration) pass the vent-specific gates? It SHOULD NOT.

**Objective.** Run the og-01..03 behavior tests on a vent-feature-ablated control; the campaign's gates must reject it, and the vent must pass the same tests.

**Hypothesis.** The og-01/og-02/og-03 gates REJECT this control; if they pass it, the gates measure generic nonequilibrium chemistry, not the vent mechanism.

**Claim.** The og-01/02/03 gates are vent-specific and falsifiable. A generic nonequilibrium control — the SAME real composites with no sustained H2, no Fe/Ni/S catalytic interface and no pore concentration — is rejected by every discriminating gate (carbon flux, formate yield, protometabolic throughput, thermophoretic Péclet, enrichment), while the vent passes all five. So a PASS elsewhere in the campaign certifies the vent MECHANISM, not merely "some nonequilibrium chemistry".


### Design — originally proposed

_The mechanisms and methods proposed when this study was designed; the model **actually built** (and the tools it uses) is shown below. These may differ as the design was realized._

**Proposed mechanisms**

- the same real og-01/02/03 composites with vent features (H2, Fe/Ni/S catalyst, pore concentration) disabled

**Proposed models / methods**

- flat gradient, inert surface, open pore, well-mixed — a generic abiotic nonequilibrium mixture
- re-run the og-01/02/03 behavior-test gates unchanged and require rejection

**Key assumptions**

- This study passes by REJECTING; a control that passes the gates would mean the gates are not vent-specific.
- The control is a parameterization of the real og-01/02/03 composites, so it tests the same models, not a mock.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `Generic non-vent control (carbon-fixation composite, vent features off)` | `viva_genesis.composites.carbon_fixation.hydrothermal_carbon_fixation` | 0 | gradient_efficiency=0.0, catalyst_active=0.0, h2=1e-09 |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `viva_genesis.composites.carbon_fixation.hydrothermal_carbon_fixation`**

A reusable H2/pH-driven carbon-fixation module. Module C = genuine FiPy Nernst-Planck pH-gradient electrochemistry (pmf, current density). Module D = genuine eQuilibrator ΔG' feasibility gate + lab-calibrated (Preiner/Hudson) mineral-catalyzed CO2 reduction to formate/acetate/pyruvate. gradient_efficiency=1.0 is the favorable arm; ~0.1 is the Tutolo-2026 in-situ collapse. catalyst_active/h2=0 give the controls. Requires equilibrator_api + fipy.


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_viva_genesis_composites_carbon_fixation_hydrothermal_carbon_fixation = {
    'gradient_efficiency': 1.0,   # Usable fraction of the geochemical ΔpH at in-situ T/P (1.0 favorable; ~0.1 Tutolo-2026 collapse; 0 = no-gradient control)
    'catalyst_active'    : 1.0,   # Active Fe/Ni/S mineral catalyst (0 = no-catalyst control)
    'h2'                 : 0.005,   # Dissolved H2 feed (M); ~0 = no-H2 control
    'co2'                : 0.002,   # Dissolved CO2 feed (M)
    'ph_vent'            : 9.5,   # Alkaline vent-side pH (from og-00)
    'ph_ocean'           : 6.0,   # Acidic early-ocean-side pH
    'dilution_rate'      : 0.0,   # Open-flow washout (1/s); 0 = closed batch (yield studies), >0 = open vent reactor → sustained-throughput regime (phase map)
}

### Typed contract (declared interface)

_The bigraph-schema typed ports + conservation predicates this study's composite must satisfy (audited in the workbench §Contract panel). Declared design — independent of whether a run exists yet._

**Inputs**

| port | type | units | bounds |
| --- | --- | --- | --- |
| `temperature` | float | kelvin | [273.0, 673.0] |
| `pH` | float | dimensionless | [0.0, 14.0] |
| `H2` | float | molar | >= 0.0 |
| `CO2` | float | molar | >= 0.0 |

**Outputs**

| port | type | units | bounds |
| --- | --- | --- | --- |
| `net_carbon_flux` | float | mole/second |  |
| `formate` | float | molar | >= 0.0 |

**Conditions**

- `charge_neutrality` (invariant): `abs(net_charge) < tol`
- `energy_directionality` (invariant): `free_energy_flux_logged == true`
- `mass_balance` (invariant): `abs(residual_C)+abs(residual_H)+abs(residual_O) < tol`


### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: og-adv-generic-nonequilibrium ===
STUDY = 'og-adv-generic-nonequilibrium'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_Generic_non_vent_control_carbon_fixation_composite_vent_features_off = 100
INTERVAL_Generic_non_vent_control_carbon_fixation_composite_vent_features_off = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_genesis_composites_carbon_fixation_hydrothermal_carbon_fixation if 'ov_viva_genesis_composites_carbon_fixation_hydrothermal_carbon_fixation' in dir() else None  # edits from the panel cell, if any
            spec_viva_genesis_composites_carbon_fixation_hydrothermal_carbon_fixation = materialize_composite('viva_genesis.composites.carbon_fixation.hydrothermal_carbon_fixation', _ov)
            _comp = Composite({'state': spec_viva_genesis_composites_carbon_fixation_hydrothermal_carbon_fixation['state']}, core=core)
            _comp.run(STEPS_Generic_non_vent_control_carbon_fixation_composite_vent_features_off)
        print('ran', 'Generic non-vent control (carbon-fixation composite, vent features off)', f'(viva_genesis.composites.carbon_fixation.hydrothermal_carbon_fixation) for', STEPS_Generic_non_vent_control_carbon_fixation_composite_vent_features_off, 'steps')
    except Exception as _e:
        print(f"Generic non-vent control (carbon-fixation composite, vent features off) not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**vent-specificity-map**


In [ ]:
# vent-specificity-map
show_viz(_render_one('local:VentSpecificityMap', {'title': 'Adversarial control — vent-specificity of the gates', 'grid_json': 'workspace/studies/og-adv-generic-nonequilibrium/artifacts/vent_specificity_grid.json'}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| gates-reject-generic _(primary)_ | kind=observable detail=run og-01/02/03 gates on the generic control; each measured value below its threshold | op controls_absent detail all five gates reject generic (net flux / formate / throughput / Péclet / enrichment below threshold) |
| vent-specificity-demonstrated _(primary)_ | kind=observable detail=vent vs generic on identical gates | op arms_reported detail vent passes 5/5 where the control fails 5/5 |


## Open decisions
- Publication milestone: close the first paper at the pre-genetic arc (og-00 -> og-03) or extend through the early-genetic arc (og-04 protocells -> og-06 heredity)?
